# Implementing an SLR-Table-Generator

## A Grammar for Grammars

We use the following grammar to describe the syntax of a context free grammar.
```
grammar
    : rule
    | rule grammar
    ;

rule
    : VARIABLE ':' body_list ';'
    ;

body_list
    : body 
    | body '|' body_list
    ;

body
    : 
    | item body
    ;
 
item : VARIABLE 
     | TOKEN  
     | LITERAL
     ;
```

As the goal is to generate an *SLR-table-generator* we first need to implement a parser for context free grammars.
The file `arith.g` in the directory `Examples` contains an example grammar that describes arithmetic expressions.

In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
printfn "%s" (System.IO.File.ReadAllText "Examples/arith.g")

We use FsLexYacc to parse grammar specifications. The shared `.fsl` and `.fsy` files are in `../Chapter-08/GrammarSyntax`. Their lexer distinguishes named terminals, nonterminals and quoted literals.

In [ ]:
#r "../Chapter-08/GrammarSyntax/bin/Debug/net9.0/FsLexYacc.Runtime.dll"
#r "../Chapter-08/GrammarSyntax/bin/Debug/net9.0/GrammarSyntax.dll"
open GrammarSyntax

The immutable `Symbol`, `Rule` and `Grammar` types come from `GrammarSyntax`.

In [ ]:
let parse = Parse.file

In [ ]:
let grammar = parse "Examples/arith.g"
printfn "%A" grammar

The grammar parser returns one typed `Rule` for each alternative, including empty right-hand sides.

F# records and discriminated unions provide readable, structurally comparable types.

### Grammar Rules

A `Rule` record is immutable and can be stored in a `Set` or used as a map key. Terminal and nonterminal union cases make the distinction explicit.

The shared compiled domain types below are immutable. Their definitions are in `../Chapter-08/GrammarSyntax/Types.fs`; aliases keep repeated notebook imports compatible.

```fsharp
type MarkedRule = { Rule: Rule; Dot: int }
type Action = Shift of int | Reduce of Rule | Accept
type ParseTable<'Item when 'Item: comparison> =
    { States: Set<'Item> array
      Actions: Map<int * string, Set<Action>>
      Gotos: Map<int * string, int> }
```

In [ ]:
// Type aliases keep the teaching names and use the same compiled domain values
// when several exported notebooks are loaded into a single F# session.
type MarkedRule = GrammarSyntax.MarkedRule
type Action = GrammarSyntax.Action
type ParseTable<'Item when 'Item: comparison> = GrammarSyntax.ParseTable<'Item>

let variables grammar = grammar.Rules |> List.map (fun rule -> rule.Head) |> Set.ofList
let terminals grammar =
    grammar.Rules |> List.collect (fun rule -> rule.Body)
    |> List.choose (function Terminal name -> Some name | _ -> None) |> Set.ofList
let isComplete (item: MarkedRule) = item.Dot = item.Rule.Body.Length
let symbolAfterDot (item: MarkedRule) = List.tryItem item.Dot item.Rule.Body
let nextVar item = match symbolAfterDot item with Some(Nonterminal name) -> Some name | _ -> None
let moveDot (item: MarkedRule) = { item with Dot = item.Dot + 1 }
let toRule (item: MarkedRule) = item.Rule

No tuple-to-object conversion is necessary: FsYacc semantic actions construct `Rule` records directly.

`Symbol` distinguishes `Nonterminal name` from `Terminal name`. No capitalization heuristic is needed after scanning.

Given a list `Rules` of `GrammarRules`, the function `variables(Rules)` returns the set of all *variables* occuring in `Rules`.

In [ ]:
printfn "%A" (variables grammar)

Given a list `Rules` of `GrammarRules`, the function `terminals(Rules)` returns the set of all *tokens* and *literals* occuring in `Rules`.

In [ ]:
printfn "%A" (terminals grammar)

## Marked Rules

The type `MarkedRule` represents the marked rule
$$ v \rightarrow \alpha \bullet \beta $$

`Rule.Head` stores $v$, `Rule.Body` stores the entire right-hand side $\alpha\beta$, and `Dot` stores the length of $\alpha$. Each symbol is a typed terminal or nonterminal. Immutable F# records provide structural equality and comparison, so marked rules can be elements of sets.

Marked rules keep the entire production and the dot position. F# record equality and ordering supply the comparison needed by sets.

Given a *marked rule* `item`, the function `isComplete` checks, whether the *marked rule* `item` has the form
$$ c \rightarrow \alpha\; \bullet,$$
i.e. it checks, whether the $\bullet$ is at the end of the grammar rule.

Given a *marked rule* `item` of the form
$$ c \rightarrow \alpha \bullet X\, \delta, $$
the function `symbolAfterDot` returns the *symbol* $X$. If there is no symbol after the $\bullet$, the method returns `None`.

Given a marked rule of the form
$$ c \rightarrow \alpha \bullet b \delta, $$
this function returns the variable $b$ following the dot.  If there is no variable following the dot, the function returns `None`.

The function `moveDot item` transforms a *marked rule*  of the form 
$$ c \rightarrow \alpha \bullet X\, \beta $$
into a *marked rule* of the form
$$ c \rightarrow \alpha\, X \bullet \beta, $$
i.e. the $\bullet$ is moved over the next symbol.  Invocation of this method assumes that there is a symbol
following the $\bullet$.

The function `toRule item` turns the *marked rule* `item` into  a `Rule`, i.e. the *marked rule*
$$ c \rightarrow \alpha \bullet \beta $$
is turned into the grammar rule
$$ c \rightarrow \alpha\, \beta. $$

## SLR-Table-Generation

`Grammar` stores the start variable and rules. The derived FIRST/FOLLOW maps, automaton and parse tables remain separate values.

Maps assign each nonterminal its current set. Sets and maps are immutable; local variables hold the next approximation during fixed-point iteration.

`computeTables` computes FIRST/FOLLOW sets, LR(0) states, transitions, actions and gotos. Every conflicting action remains visible in its set of candidates.

Rules are structural records. The emitted table therefore carries each reduction rule directly rather than assigning extra rule names.

`computeFirst grammar` computes the FIRST map by fixed-point iteration. We include $\lambda$ exactly when a nonterminal derives the empty word:
$$\mathrm{First}(a)=\{t\in T\mid\exists\gamma:\ a\Rightarrow^*t\gamma\}\ \cup\ \{\lambda\mid a\Rightarrow^*\lambda\}.$$
For a list of grammar symbols:
- $\mathrm{FirstList}(\lambda)=\{\lambda\}$.
- $\mathrm{FirstList}(t\beta)=\{t\}$ for terminal $t$.
- $\mathrm{FirstList}(a\beta)=\mathrm{First}(a)$ if $\lambda\notin\mathrm{First}(a)$.
- Otherwise $\mathrm{FirstList}(a\beta)=(\mathrm{First}(a)\setminus\{\lambda\})\cup\mathrm{FirstList}(\beta)$.
For productions $a\rightarrow\alpha_1\mid\cdots\mid\alpha_n$ we repeatedly enlarge
$$\mathrm{First}(a)=\bigcup_{i=1}^{n}\mathrm{FirstList}(\alpha_i).$$
The empty string `""` represents $\lambda$ in the implementation.

In [ ]:
// The empty string represents epsilon in FIRST sets.
let rec firstList first symbols =
    match symbols with
    | [] -> Set.singleton ""
    | Terminal token :: _ -> Set.singleton token
    | Nonterminal name :: tail ->
        let head = Map.find name first
        if Set.contains "" head then Set.union (Set.remove "" head) (firstList first tail)
        else head

let computeFirst grammar =
    let mutable first = variables grammar |> Seq.map (fun v -> v, Set.empty) |> Map.ofSeq
    let mutable changed = true
    while changed do
        let previous = first
        for rule in grammar.Rules do
            first <- Map.add rule.Head (Set.union first[rule.Head] (firstList first rule.Body)) first
        changed <- first <> previous
    first

let epsUnion left right =
    if Set.contains "" left then Set.union (Set.remove "" left) right else left

Given a list of grammar symbols `alpha`, the function `firstList first alpha` computes the function $\texttt{FirstList}(\alpha)$ that has been defined above.  If `alpha` is *nullable*, then the result will contain the empty string $\lambda$.

The arguments `S` and `T` of `epsUnion` are sets that contain tokens and, additionally, they might contain the empty string 
$\lambda =$ `''`.  The specification of `epsUnion` is:
$$ \mathtt{eps\_union}(S, T) = \left\{ \begin{array}{ll}
                                       S          & \mbox{if $\lambda \not\in S$} \\
                                       S \cup T   & \mbox{if $\lambda \in S \wedge \lambda \in T$} \\
                                       S \cup T - \{\lambda \} & \mbox{if $\lambda \in S \wedge \lambda \not\in T$}
                                      \end{array}
                              \right.
$$

Given an augmented grammar $G = \langle V,T,R\cup\{\widehat{s} \rightarrow s\,\$\}, \widehat{s}\rangle$ 
and a variable $a$, the set of tokens that might follow $a$ is defined as:
$$\texttt{Follow}(a) := 
 \bigl\{ t \in \widehat{T} \,\bigm|\, \exists \beta,\gamma \in (V \cup \widehat{T})^*: 
                           \widehat{s} \Rightarrow^* \beta \,a\, t\, \gamma 
  \bigr\}.
$$
The function `computeFollow` computes the sets $\texttt{Follow}(a)$ for all variables $a$ via a *fixed-point iteration*.

In [ ]:
let computeFollow grammar first =
    let mutable follow = variables grammar |> Seq.map (fun v -> v, Set.empty) |> Map.ofSeq
    follow <- Map.add grammar.Start (Set.singleton "$") follow
    let mutable changed = true
    while changed do
        let previous = follow
        for rule in grammar.Rules do
            rule.Body |> List.iteri (fun position symbol ->
                match symbol with
                | Terminal _ -> ()
                | Nonterminal name ->
                    let tailFirst = firstList first (List.skip (position + 1) rule.Body)
                    let additions = epsUnion tailFirst follow[rule.Head]
                    follow <- Map.add name (Set.union follow[name] additions) follow)
        changed <- previous <> follow
    follow

If $\mathcal{M}$ is a set of *marked rules*, then the *closure* of $\mathcal{M}$ is the smallest set $\mathcal{K}$ such that
we have the following:
- $\mathcal{M} \subseteq \mathcal{K}$,
- If $a \rightarrow \beta \bullet c\, \delta$ is a *marked rule* from 
  $\mathcal{K}$, and $c$ is a variable and if, furthermore,
  $c \rightarrow \gamma$ is a grammar rule,
  then the marked rule $c \rightarrow \bullet \gamma$
  is an element of $\mathcal{K}$:
  $$(a \rightarrow \beta \bullet c\, \delta) \in \mathcal{K} 
         \;\wedge\; 
         (c \rightarrow \gamma) \in R
         \;\Rightarrow\; (c \rightarrow \bullet \gamma) \in \mathcal{K}
  $$

We define $\texttt{closure}(\mathcal{M}) := \mathcal{K}$.  The function `closure` computes this closure for a given set of *marked rules* via a *fixed-point iteration*.

In [ ]:
let closure grammar items =
    let mutable result = items
    let mutable changed = true
    while changed do
        let additions =
            result |> Seq.choose nextVar
            |> Seq.collect (fun name ->
                grammar.Rules |> Seq.filter (fun rule -> rule.Head = name)
                |> Seq.map (fun rule -> { Rule = rule; Dot = 0 })) |> Set.ofSeq
        let updated = Set.union result additions
        changed <- updated <> result
        result <- updated
    result

let goto grammar items symbol =
    items |> Seq.filter (fun item -> symbolAfterDot item = Some symbol)
    |> Seq.map moveDot |> Set.ofSeq |> closure grammar

Given a set of *marked rules* $\mathcal{M}$ and a *grammar symbol* $X$, the function $\texttt{goto}(\mathcal{M}, X)$ 
is defined as follows:
$$\texttt{goto}(\mathcal{M}, X) := \texttt{closure}\Bigl( \bigl\{ 
   a \rightarrow \beta\, X \bullet \delta \bigm| (a \rightarrow \beta \bullet X\, \delta) \in \mathcal{M} 
   \bigr\} \Bigr).
$$

The function `allStates` computes the set of all states of an *SLR-parser*.  The function starts with the state
$$ \texttt{closure}\bigl(\{ \widehat{s} \rightarrow \bullet s \, $\}\bigr) $$
and then tries to compute new states by using the function `goto`.  This computation proceeds via a 
*fixed-point iteration*.  Once all states have been computed, the function assigns names to these states.
The array index is the state number; the discovery map assigns the same number to every structurally equal state.

In [ ]:
// Breadth-first discovery and structural ordering produce reproducible state numbers.
let allStates grammar =
    let augmented = { Head = "ŝ"; Body = [Nonterminal grammar.Start] }
    let start = closure grammar (Set.singleton { Rule = augmented; Dot = 0 })
    let states = ResizeArray<Set<MarkedRule>>()
    let names = System.Collections.Generic.Dictionary<Set<MarkedRule>, int>()
    let mutable transitions = Map.empty
    states.Add start
    names.Add(start, 0)
    let mutable position = 0
    while position < states.Count do
        let symbols = states[position] |> Seq.choose symbolAfterDot |> Set.ofSeq
        for symbol in symbols do
            let target = goto grammar states[position] symbol
            let targetName =
                match names.TryGetValue target with
                | true, name -> name
                | _ ->
                    let name = states.Count
                    states.Add target
                    names.Add(target, name)
                    name
            transitions <- Map.add (position, symbol) targetName transitions
        position <- position + 1
    states.ToArray(), transitions

The following function computes the *action table* and is defined as follows:
- If $\mathcal{M}$ contains a *marked rule* of the form $a \rightarrow \beta \bullet t\, \delta$
  then we have
  $$\texttt{action}(\mathcal{M},t) := \langle \texttt{shift}, \texttt{goto}(\mathcal{M},t) \rangle.$$
- If $\mathcal{M}$ contains a marked rule of the form $a \rightarrow \beta\, \bullet$ and we have
  $t \in \texttt{Follow}(a)$, then we define
  $$\texttt{action}(\mathcal{M},t) := \langle \texttt{reduce}, a \rightarrow \beta \rangle$$
- If $\mathcal{M}$ contains the marked rule $\widehat{s} \rightarrow s \bullet \$ $, then we define 
  $$\texttt{action}(\mathcal{M},\$) := \texttt{accept}. $$
- Otherwise, we have
  $$\texttt{action}(\mathcal{M},t) := \texttt{error}. $$

In [ ]:
let addAction key action actions =
    let candidates = Map.tryFind key actions |> Option.defaultValue Set.empty
    Map.add key (Set.add action candidates) actions

let computeTables grammar =
    let first = computeFirst grammar
    let follow = computeFollow grammar first
    let states, transitions = allStates grammar
    let mutable actions = Map.empty
    let mutable gotos = Map.empty
    for KeyValue((state, symbol), target) in transitions do
        match symbol with
        | Terminal token -> actions <- addAction (state, token) (Shift target) actions
        | Nonterminal name -> gotos <- Map.add (state, name) target gotos
    for state in 0 .. states.Length - 1 do
        for item in states[state] do
            if isComplete item then
                if item.Rule.Head = "ŝ" then
                    actions <- addAction (state, "$") Accept actions
                else
                    for token in follow[item.Rule.Head] do
                        actions <- addAction (state, token) (Reduce item.Rule) actions
    { States = states; Actions = actions; Gotos = gotos }

let conflicts table =
    table.Actions |> Map.toList |> List.filter (fun (_, candidates) -> Set.count candidates > 1)

let describeItem item =
    let symbol = function Nonterminal name -> name | Terminal name -> sprintf "'%s'" name
    let before = item.Rule.Body |> List.take item.Dot |> List.map symbol
    let after = item.Rule.Body |> List.skip item.Dot |> List.map symbol
    sprintf "%s -> %s • %s" item.Rule.Head (String.concat " " before) (String.concat " " after)

let printTable describe table =
    table.States |> Array.iteri (fun state items ->
        printfn "s%d = {%s}" state (items |> Seq.map describe |> String.concat "; "))
    for KeyValue((state, token), actions) in table.Actions do printfn "action(s%d, %s) = %A" state token actions
    for KeyValue((state, name), target) in table.Gotos do printfn "goto(s%d, %s) = s%d" state name target
    for key, actions in conflicts table do printfn "CONFLICT %A: %A" key actions

The function `computeTables` computes the *goto table*.

In [ ]:
let table = computeTables grammar
printTable describeItem table
checkNotebook ((conflicts table).IsEmpty)

In [ ]:
// Export a self-contained F# table, including the state descriptions.
let dumpParseTable path describe table =
    let symbol = function Nonterminal name -> sprintf "Nonterminal %A" name | Terminal name -> sprintf "Terminal %A" name
    let rule rule = sprintf "{ Head = %A; Body = [%s] }" rule.Head (rule.Body |> List.map symbol |> String.concat "; ")
    let action = function Shift state -> sprintf "Shift %d" state | Reduce r -> "Reduce (" + rule r + ")" | Accept -> "Accept"
    let actions =
        table.Actions |> Map.toList |> List.map (fun ((state, token), candidates) ->
            sprintf "    (%d, %A), Set.ofList [%s]" state token (candidates |> Seq.map action |> String.concat "; "))
    let gotos = table.Gotos |> Map.toList |> List.map (fun ((state, name), target) -> sprintf "    (%d, %A), %d" state name target)
    let states =
        table.States |> Array.map (fun items ->
            sprintf "    Set.ofList [%s]" (items |> Seq.map (describe >> sprintf "%A") |> String.concat "; ")) |> Array.toList
    let text =
        String.concat "\n"
            (["#load \"SLR.fsx\""; "open SLR"; "open GrammarSyntax"; "let actionTable = Map.ofList ["] @ actions @
             ["]"; "let gotoTable = Map.ofList ["] @ gotos @
             ["]"; "let states = [|"] @ states @
             ["|]"; "let table : ParseTable<string> = { States = states; Actions = actionTable; Gotos = gotoTable }"])
    System.IO.File.WriteAllText(path, text)

let analyse file =
    let grammar = Parse.file file
    let table = computeTables grammar
    printTable describeItem table
    table

In [ ]:
dumpParseTable "parse-table.fsx" describeItem table

In [ ]:
printfn "%s" (System.IO.File.ReadAllText "parse-table.fsx")

In [ ]:
analyse "Examples/postfix.g" |> ignore

In [ ]:
analyse "Examples/arith-small.g" |> ignore

In [ ]:
checkNotebook (not ((conflicts (analyse "Examples/arith-ambiguous.g")).IsEmpty))

In [ ]:
analyse "Examples/c-grammar-slr.g" |> ignore

In [ ]:
checkNotebook (not ((conflicts (analyse "Examples/c-grammar.g")).IsEmpty))

In [ ]:
let nullable = Parse.text "s: a 'x'; a: ;"
checkNotebook (Set.contains "" (computeFirst nullable).["a"])
checkNotebook (Set.contains "x" (computeFollow nullable (computeFirst nullable)).["a"])

The mathematical descriptions sometimes write the augmented production as $\widehat{s}\rightarrow s\,\$$. The implementation uses the equivalent convention $\widehat{s}\rightarrow s$, keeping `$` exclusively as the external end-of-input lookahead. The unmodified C fixture remains ambiguous around dangling `else`; its remaining LR conflicts are reported rather than resolved silently.